In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# --------------------------------------------------
# 1. Load dataset
# --------------------------------------------------
df = pd.read_csv("../data/merged_dataset_organized_47photic.csv")

# --------------------------------------------------
# 1B. Extract metadata from NaN-depth rows (one per cast)
# --------------------------------------------------
metadata_cols = ["CAST_COUNT", "Cruise_ID", "Cruz_Sta", "Cast_ID", "Sta_ID",
                 "Distance", "Date", "Time", "Lat_Dec", "Lon_Dec",
                 "Ac_Line", "Bottom_D", "Secchi", "IntChl", "IntC14",
                 "TimeZone", "Visibility"]

# Get metadata from rows where Secchi is not null (the secchi measurement rows)
metadata = (
    df[df["Secchi"].notna()][metadata_cols]
    .drop_duplicates("CAST_COUNT")
)

# --------------------------------------------------
# 2. Define baseline depth
# --------------------------------------------------
BASELINE_DEPTH = 2
SUMMING_DEPTH = 20  # Sum from 2m to 20m (or photic depth, whichever is shallower)

photic_depths = df[df["PHOTIC_ZONE"] == True][["CAST_COUNT", "DEPTH"]].rename(
    columns={"DEPTH": "PHOTIC_DEPTH"}
).drop_duplicates("CAST_COUNT")

print(f"Photic depths found for {len(photic_depths)} casts")

# Join photic depth onto full dataframe
df_with_photic = df.merge(photic_depths, on="CAST_COUNT", how="left")

print("\nCalculating summed variables...")

# Filter to baseline → min(20m, photic_depth) range per cast
df_range = df_with_photic[
    (df_with_photic["DEPTH"] >= BASELINE_DEPTH) &
    (df_with_photic["DEPTH"] <= df_with_photic["PHOTIC_DEPTH"].clip(upper=SUMMING_DEPTH))
]

print(f"Rows in summing range: {len(df_range)}")

# Sum variables over this range
summed = df_range.groupby("CAST_COUNT").agg(
    XMISS_SUMMED=("XMISS", "sum"),
    CHL_A_SUMMED=("CHL_A", "sum"),
    PHAEO_SUMMED=("PHAEO", "sum"),
    ESTCHL_SUMMED=("ESTCHL_STACORR", "sum"),
    BAT_SUMMED=("BAT", "sum")
).reset_index()

print(f"Summed variables calculated for {len(summed)} casts")

# ============================================================================
# KEEP ONLY PHOTIC_ZONE == TRUE ROWS
# ============================================================================

print("\nFiltering to photic zone rows only...")
df_photic = df[df["PHOTIC_ZONE"] == True].copy()

print(f"Photic zone rows: {len(df_photic)}")

# ============================================================================
# MERGE SUMMED VARIABLES ONTO PHOTIC ROWS
# ============================================================================

print("\nMerging summed variables...")
df_final = df_photic.merge(summed, on="CAST_COUNT", how="left")

# ============================================================================
# MERGE METADATA (clean version from Secchi rows)
# ============================================================================

print("Merging metadata...")

# Drop the potentially sparse metadata columns from photic rows
metadata_to_drop = [c for c in metadata_cols[1:] if c in df_final.columns]
df_final = df_final.drop(columns=metadata_to_drop)

# Merge clean metadata
df_final = df_final.merge(metadata, on="CAST_COUNT", how="left")

# --------------------------------------------------
# 7A. Two-point Beer-Lambert K_PAR (original formula)
# --------------------------------------------------

# Baseline PAR at 2m
baseline_par = df[df["DEPTH"] == BASELINE_DEPTH][
    ["CAST_COUNT", "PAR"]
].rename(columns={"PAR": "PAR_BASELINE"}).drop_duplicates("CAST_COUNT")

df_final = df_final.merge(baseline_par, on="CAST_COUNT", how="left")

df_final["K_PAR"] = -np.log(
    df_final["PAR"] / df_final["PAR_BASELINE"]
) / df_final["DEPTH"]

df_final = df_final.drop(columns=["PAR_BASELINE"])  # remove intermediate column


# --------------------------------------------------
# 7B. Regression slope-based K_PAR_SLOPE
# --------------------------------------------------

kpar_slope_results = []

for cast_id, cast_df in df.groupby("CAST_COUNT"):

    # Get photic depth
    photic_row = cast_df[cast_df["PHOTIC_ZONE"] == True]
    if photic_row.empty:
        continue

    photic_depth = photic_row["DEPTH"].values[0]

    # Subset depths between 2m and photic depth
    subset = cast_df[
        (cast_df["DEPTH"] >= BASELINE_DEPTH) &
        (cast_df["DEPTH"] <= photic_depth)
    ].copy()

    subset = subset[subset["PAR"] > 0]

    if len(subset) < 2:
        continue

    subset["LOG_PAR"] = np.log(subset["PAR"])

    X = subset[["DEPTH"]].values
    y = subset["LOG_PAR"].values

    model = LinearRegression()
    model.fit(X, y)

    slope = model.coef_[0]

    kpar_slope_results.append({
        "CAST_COUNT": cast_id,
        "K_PAR_SLOPE": -slope
    })

kpar_slope_df = pd.DataFrame(kpar_slope_results)

df_final = df_final.merge(kpar_slope_df, on="CAST_COUNT", how="left")

# --------------------------------------------------
# 8. Save final dataset
# --------------------------------------------------
df_final.to_parquet("../data/Parquet/New_estchlsummed_batsummed.parquet", index=False)

print(df_final.head())

FileNotFoundError: [Errno 2] No such file or directory: '../data/merged_dataset_organized_47photic.csv'

In [8]:
# 1. How much of each metadata column is actually populated?
print(df[["Cruise_ID", "Cruz_Sta", "Cast_ID", "Secchi", "IntChl"]].isnull().sum())

# 2. Check if they use a fill value like 0, -9999, or empty string instead of NaN
print(df["Cruise_ID"].value_counts(dropna=False).head(10))

# 3. See what the raw data looks like for one cast
print(df[df["CAST_COUNT"] == df["CAST_COUNT"].iloc[0]][
    ["CAST_COUNT", "DEPTH", "Cruise_ID", "Secchi", "IntChl"]
])

Cruise_ID    1253192
Cruz_Sta     1253192
Cast_ID      1253192
Secchi       1253192
IntChl       1253203
dtype: int64
Cruise_ID
NaN                  1253192
2015-01-15-C-32NM         39
2014-07-06-C-32NM         37
2020-07-13-C-33SR         37
2009-07-14-C-31M4         37
2010-01-13-C-32NM         37
1996-08-07-C-32NM         37
2011-01-13-C-32NM         35
1995-07-06-C-31JD         35
1994-08-05-C-32NM         34
Name: count, dtype: int64
    CAST_COUNT  DEPTH          Cruise_ID  Secchi  IntChl
0        27453    2.0                NaN     NaN     NaN
1        27453    3.0                NaN     NaN     NaN
2        27453    4.0                NaN     NaN     NaN
3        27453    5.0                NaN     NaN     NaN
4        27453    6.0                NaN     NaN     NaN
5        27453    7.0                NaN     NaN     NaN
6        27453    8.0                NaN     NaN     NaN
7        27453    9.0                NaN     NaN     NaN
8        27453   10.0                NaN   